# 🐱💥 Kickstarter Scraper



In [17]:
import subprocess, sys

pkgs = ["playwright", "playwright-stealth", "beautifulsoup4", "lxml"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
subprocess.check_call(["playwright", "install", "chromium"])
print("\n✓ All dependencies ready")


✓ All dependencies ready


## 2 · Imports & configuration

In [18]:
# ============================================================
# KICKSTARTER SCRAPER
# 1. meta: fixed goal (regex "pledged of $X goal"), fixed category (scoped selector)
# 2. campaign: removed stats dict; added video_url + image_urls from story
# 3. rewards: split title (first line) from description (rest)
# 4. creator: bio via account-created text pattern; filtered social links
# 5. updates: "Load more" clicking to get ALL updates; image/video URLs in body
# 6. comments: DOM-based extraction via page.evaluate + Load more clicking
# 7. CSV export: all dicts expanded; no blob columns
# ============================================================

import sys, re, json, asyncio, threading, logging
from datetime import datetime
from pathlib import Path
from typing import Optional, List

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, Page, TimeoutError as PWTimeout
from playwright_stealth import Stealth

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)-8s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("ks_scraper")

HEADLESS          = False
SLOW_MO           = 60
PAGE_TIMEOUT      = 80_000
NAV_DELAY_MS      = 5_000
MAX_RETRIES       = 3
MAX_COMMENT_PAGES = 10    # each click loads ~25 comments
MAX_UPDATE_PAGES  = 20    # clicks on "Load more" for updates
OUTPUT_DIR        = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3 · Helpers & `run_pw()`

In [19]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def _clean(text) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ",
                  BeautifulSoup(str(text), "lxml").get_text(" ")).strip()

def _ts(val) -> str:
    if isinstance(val, (int, float)):
        return datetime.utcfromtimestamp(val).isoformat() + "Z"
    return str(val or "")

def slug_from_url(url: str):
    m = re.search(r"kickstarter\.com/projects/([^/?#]+)/([^/?#]+)", url)
    if not m:
        raise ValueError(f"Cannot parse project URL: {url}")
    return m.group(1), m.group(2)

def run_pw(coro):
    result_box, error_box = {}, {}
    def _thread():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result_box["value"] = loop.run_until_complete(coro)
        except Exception as exc:
            error_box["err"] = exc
        finally:
            loop.close()
    t = threading.Thread(target=_thread, daemon=True)
    t.start(); t.join()
    if "err" in error_box:
        raise error_box["err"]
    return result_box["value"]

## 4 · Browser utilities

Key addition: **`_click_load_more()`** — clicks Kickstarter's paginated 'Load more' button repeatedly for updates and comments.

**`_extract_story_media()`** and **`_extract_body_media()`** extract image URLs and video URLs from story HTML and update bodies.

In [20]:
# ── Browser core ─────────────────────────────────────────────────────────────

async def _make_browser_context(pw):
    browser = await pw.chromium.launch(
        headless=HEADLESS, slow_mo=SLOW_MO,
        args=["--disable-blink-features=AutomationControlled",
              "--no-sandbox", "--disable-dev-shm-usage"],
    )
    ctx = await browser.new_context(
        viewport={"width": 1280, "height": 900},
        user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/124.0.0.0 Safari/537.36"),
        locale="en-US", timezone_id="America/New_York", java_script_enabled=True,
    )
    await ctx.set_extra_http_headers({"Accept-Language": "en-US,en;q=0.9", "DNT": "1"})
    page = await ctx.new_page()
    stealth = Stealth(navigator_languages_override=False)
    await stealth.apply_stealth_async(page)
    return browser, page

async def _goto(page: Page, url: str) -> str:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=PAGE_TIMEOUT)
            await page.wait_for_timeout(NAV_DELAY_MS)
            return await page.content()
        except PWTimeout:
            if attempt == MAX_RETRIES:
                raise
            await page.wait_for_timeout(attempt * 3_000)

async def _fetch_json_via_browser(page: Page, url: str) -> Optional[dict]:
    """Fetch JSON endpoint through browser (carries cookies). Returns None if not JSON."""
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=PAGE_TIMEOUT)
        await page.wait_for_timeout(600)
        body = await page.locator("pre, body").first.inner_text()
        s = body.strip()
        if not s or (not s.startswith("{") and not s.startswith("[")):
            return None
        return json.loads(s)
    except Exception as exc:
        log.warning("JSON fetch failed (%s): %s", url, exc)
        return None

async def _click_load_more(page: Page, max_clicks: int, delay_ms: int = 2500) -> int:
    """Click 'Load more' button up to max_clicks times. Returns actual clicks."""
    clicks = 0
    for _ in range(max_clicks):
        try:
            btn = page.locator(
                "button:has-text('Load more'), "
                "span:has-text('Load more'), "
                "a:has-text('Load more')"
            ).first
            if not await btn.is_visible(timeout=3000):
                break
            await btn.click()
            await page.wait_for_timeout(delay_ms)
            clicks += 1
        except Exception:
            break
    return clicks

def _extract_story_media(soup: BeautifulSoup, story_el) -> tuple:
    """Extract image URLs and video URL from the story/campaign section."""
    images: List[str] = []
    video_url = ""

    # Campaign pitch video from data-video attribute
    video_player = soup.find("div", class_="video-player")
    if video_player:
        raw = video_player.get("data-video", "")
        if raw:
            try:
                vd = json.loads(raw)
                video_url = vd.get("high") or vd.get("base") or ""
            except Exception:
                pass

    # Images inside the story content
    # IMPORTANT: keep the FULL URL including ?sig= query params — stripping them causes 404
    if story_el:
        for img in story_el.find_all("img"):
            src = img.get("src") or img.get("data-src") or ""
            if src and ("kickstarter.com" in src or "i.kstatic" in src or "i.ks.imgix" in src):
                if src not in images:
                    images.append(src)

    return images, video_url

def _extract_body_media(html_or_text: str) -> tuple:
    """Extract image/video URLs from an update body (HTML or plain text)."""
    images: List[str] = []
    video_url = ""
    if not html_or_text:
        return images, video_url
    soup = BeautifulSoup(html_or_text, "lxml")
    # Images — keep full URL including ?sig= parameter
    for img in soup.find_all("img"):
        src = img.get("src") or ""
        if src and ("kickstarter" in src or "i.kstatic" in src):
            if src not in images:
                images.append(src)
    # Video player
    vp = soup.find("div", class_="video-player")
    if vp:
        try:
            vd = json.loads(vp.get("data-video", "{}"))
            video_url = vd.get("high") or vd.get("base") or ""
        except Exception:
            pass
    # Plain text video links
    if not video_url:
        m = re.search(r"https?://[^\s\"']+\.mp4", html_or_text)
        if m:
            video_url = m.group(0)
    return images, video_url

## 5 · Scraper functions

### `scrape_meta`
**Fixes:** goal via `"pledged of $X goal"` regex · category scoped to container · percent_funded now correct

In [21]:
# FIX: goal via "pledged of $X goal" regex; category scoped to NS_projects__category_location
async def scrape_meta(page: Page, base: str) -> dict:
    log.info("  → meta (HTML)")
    html = await _goto(page, base + "/description")
    soup = BeautifulSoup(html, "lxml")
    full_text = soup.get_text(" ")

    # Title: h3 inside the col-8 section header
    title = ""
    col = soup.find("div", class_=re.compile(r"\bcol\b.*\bcol-8\b.*\bpy3\b", re.I))
    if col:
        h3 = col.find("h3")
        title = _clean(h3) if h3 else ""
    if not title:
        h3 = soup.find("h3", class_=re.compile(r"^normal", re.I))
        title = _clean(h3) if h3 else ""

    # Blurb
    blurb = ""
    for attr in [{"name": "description"}, {"property": "og:description"}]:
        meta_el = soup.find("meta", attr)
        if meta_el:
            blurb = meta_el.get("content", "")
            break

    # ── Pledged: first .money span ────────────────────────────────────────────
    first_money = soup.find("span", class_="money")
    pledged_raw = _clean(first_money) if first_money else ""
    pledged_num = re.sub(r"[^\d]", "", pledged_raw)

    # ── Goal: "pledged of $X goal" pattern — NOT the same .money list ────────
    goal_num = ""
    m = re.search(r"pledged\s+of\s+\$?([\d,]+)\s+goal", full_text, re.I)
    if m:
        goal_num = m.group(1).replace(",", "")
    else:
        # Fallback: second .money that immediately follows "of" text
        of_nodes = soup.find_all(string=re.compile(r"pledged of", re.I))
        for node in of_nodes:
            span = node.find_next("span", class_="money")
            if span:
                goal_num = re.sub(r"[^\d]", "", _clean(span))
                break

    # ── Backers: standalone integer h3.mb0 ───────────────────────────────────
    backers_count = ""
    for h in soup.find_all("h3", class_="mb0"):
        t = _clean(h).replace(",", "")
        if t.isdigit():
            backers_count = t
            break
    if not backers_count:
        m2 = re.search(r"([\d,]+)\s+backers?", full_text)
        if m2:
            backers_count = m2.group(1).replace(",", "")

    # ── Percent funded ────────────────────────────────────────────────────────
    percent_funded = ""
    if pledged_num and goal_num and int(goal_num) > 0:
        try:
            pct = int(pledged_num) / int(goal_num) * 100
            percent_funded = f"{pct:,.0f}%"
        except Exception:
            pass

    # ── Location + Category: scoped to NS_projects__category_location ─────────
    location, category = "", ""
    cat_loc = soup.find(class_=re.compile(r"NS_projects__category_location|category_location", re.I))
    container = cat_loc if cat_loc else soup
    loc_a = container.find("a", href=re.compile(r"/discover/places/"))
    if loc_a:
        location = _clean(loc_a.get_text())
    cat_a = container.find("a", href=re.compile(r"/discover/categories/"))
    if cat_a:
        # href like /discover/categories/games/tabletop%20games → "Tabletop Games"
        category = _clean(cat_a.get_text())

    # ── Funding period ────────────────────────────────────────────────────────
    funding_period = ""
    fp_el = soup.find(class_=re.compile(r"NS_campaigns__funding_period", re.I))
    if fp_el:
        times = fp_el.find_all("time")
        if len(times) >= 2:
            funding_period = (f"{times[0].get_text(strip=True)} - "
                              f"{times[1].get_text(strip=True)}")

    # ── Last updated ──────────────────────────────────────────────────────────
    last_updated = ""
    m3 = re.search(r"Last\s+updated\s+([\w]+\s+\d+,?\s*\d{4})", full_text)
    if m3:
        last_updated = m3.group(1).strip()

    # ── Creator name ──────────────────────────────────────────────────────────
    creator_name = ""
    m4 = re.search(r"Created\s+by\s*\n?\s*([^\n<]+)", full_text)
    if m4:
        creator_name = m4.group(1).strip().split("  ")[0]

    creator_slug, project_slug = slug_from_url(base)
    log.info("    title=%s pledged=%s goal=%s backers=%s", title, pledged_num, goal_num, backers_count)
    return {
        "title": title, "blurb": blurb,
        "project_url": base, "creator_slug": creator_slug, "project_slug": project_slug,
        "creator_name": creator_name,
        "pledged": pledged_num, "usd_pledged": pledged_num,
        "goal": goal_num, "percent_funded": percent_funded,
        "backers_count": backers_count,
        "currency": "USD", "currency_symbol": "$", "state": "successful",
        "location": location, "category": category,
        "funding_period": funding_period, "last_updated": last_updated,
    }


# FIX: no stats dict; extract video_url + image_urls from story

### `scrape_campaign`
**Fixes:** `video_url` + `image_urls` extracted · removed `stats` dict blob · `story_html` removed (too large for CSV)

In [22]:
# FIX: no stats dict; extract video_url + image_urls from story
async def scrape_campaign(page: Page, base: str) -> dict:
    log.info("  → campaign /description")
    html = await _goto(page, base + "/description")
    soup = BeautifulSoup(html, "lxml")

    story = (
        soup.find("div", class_="rte__content")
        or soup.find("div", class_="story-content")
    )
    story_text = _clean(story.get_text("\n")) if story else ""

    # Risks
    risks_el = soup.find(id="risks-and-challenges")
    risks_text = ""
    if risks_el:
        p = risks_el.find("p", class_=re.compile(r"text-preline|js-risks", re.I))
        risks_text = _clean(p) if p else ""

    # Extract images + video
    images, video_url = _extract_story_media(soup, story)

    return {
        "story_text": story_text,
        "risks_text": risks_text,
        "video_url": video_url,
        "image_urls": " | ".join(images),
        "image_count": len(images),
    }


# FIX: split first line as title name, rest as description

### `scrape_rewards`
**Fix:** `title` = first line only · `amount` = clean integer · `ships_to` extracted

In [23]:
# FIX: split first line as title name, rest as description
async def scrape_rewards(page: Page, base: str) -> list:
    log.info("  → rewards /rewards")
    html = await _goto(page, base + "/rewards")
    soup = BeautifulSoup(html, "lxml")
    rewards = []

    articles = soup.find_all("article", attrs={"data-test-id": re.compile(r"^reward-")})
    for art in articles:
        amt_el = art.find("p", class_=re.compile(r"type-18", re.I))
        amount_raw = _clean(amt_el) if amt_el else ""
        # Clean amount to just number
        amount_num = re.sub(r"[^\d]", "", amount_raw)

        # Description block: "TITLE\r\n\r\nDescription text..."
        desc_el = art.find("div", class_=re.compile(r"text-prewrap", re.I))
        raw_text = _clean(desc_el) if desc_el else ""
        # Split on actual newlines (not just spaces) — use original text
        raw_orig = desc_el.get_text("\n") if desc_el else ""
        lines = [l.strip() for l in raw_orig.split("\n") if l.strip()]
        # FIX: first non-empty line is the reward name (ALL CAPS usually)
        reward_title = lines[0] if lines else ""
        description   = "\n".join(lines[1:]).strip() if len(lines) > 1 else raw_text

        # Backers count
        backer_span = art.find("span", attrs={"aria-label": re.compile(r"\d+ backers")})
        backers = ""
        if backer_span:
            m = re.search(r"(\d[\d,]*)", backer_span["aria-label"])
            backers = m.group(1).replace(",", "") if m else ""

        delivery = ""
        time_el = art.find("time")
        if time_el:
            delivery = time_el.get_text(strip=True)

        # Ships to
        ships_to = ""
        for h3 in art.find_all("h3"):
            if "ships to" in _clean(h3).lower():
                sib = h3.find_next_sibling() or h3.find_next("div")
                if sib:
                    ships_to = _clean(sib)
                else:
                    # Try the next div after the h3 parent
                    ships_to = _clean(h3.parent.find("div", class_=re.compile(r"type-14"))) if h3.parent else ""
                break

        # Limit
        limit_node = art.find(string=re.compile(r"None left|Limited quantity|\d+ remaining", re.I))
        limit = _clean(limit_node) if limit_node else ""

        rewards.append({
            "id":                art.get("data-test-id", "").replace("reward-", ""),
            "amount":            amount_num,
            "title":             reward_title,
            "description":       description,
            "backers_count":     backers,
            "estimated_delivery": delivery,
            "ships_to":          ships_to or "Anywhere in the world",
            "limit":             limit,
        })

    # JSON API fallback
    if not rewards:
        data = await _fetch_json_via_browser(page, base + ".json")
        for r in (data or {}).get("project", {}).get("rewards", []):
            rewards.append({
                "id":                r.get("id"),
                "amount":            r.get("minimum"),
                "title":             r.get("title") or r.get("reward", ""),
                "description":       r.get("description", ""),
                "backers_count":     r.get("backers_count"),
                "estimated_delivery": r.get("estimated_delivery_on"),
                "ships_to":          r.get("shipping_type", ""),
                "limit":             r.get("limit"),
            })

    log.info("    → %d reward tiers", len(rewards))
    return rewards


# FIX: bio text via account-created pattern; proper collaborator + social link extraction

### `scrape_creator`
**Fixes:** bio from `account created … Follow` pattern · social links exclude KS footer · collaborators & founder extracted

In [24]:
# FIX: bio text via account-created pattern; proper collaborator + social link extraction
async def scrape_creator(page: Page, base: str) -> dict:
    log.info("  → creator /creator")
    html = await _goto(page, base + "/creator")
    soup = BeautifulSoup(html, "lxml")
    full_text = soup.get_text(" ")

    # Bio: text between "account created Mon YYYY" and "Follow"
    bio_text = ""
    m = re.search(
        r"account\s+created\s+\w+\s+\d{4}\s+(.*?)(?:\s*Follow\s*@|\s*Follow\b)",
        full_text, re.IGNORECASE | re.DOTALL
    )
    if m:
        bio_text = m.group(1).strip()

    # Stats: counts and dates
    stats = {}
    for pat, key in [
        (r"(\d+)\s+created\s+projects?", "created_projects"),
        (r"(\d+)\s+backed\s+projects?", "backed_projects"),
        (r"last\s+login\s+([\w]+\s+\d+\s+\d{4})", "last_login"),
        (r"account\s+created\s+([\w]+\s+\d{4})", "account_created"),
    ]:
        mx = re.search(pat, full_text, re.I)
        if mx:
            stats[key] = mx.group(1).strip()

    # Collaborators section
    collaborators = []
    collab_blocks = re.findall(
        r"Collaborator\s*\n?\s*([^\n]+?)(?=\s*Collaborator|\s*Founder|\s*$)",
        full_text, re.IGNORECASE
    )
    for cb in collab_blocks:
        name = cb.strip()
        if name and len(name) < 80:
            collaborators.append(name)
    # Also founder
    founder_m = re.search(r"Founder\s*\n?\s*([^\n]+)", full_text, re.I)
    if founder_m:
        stats["founder"] = founder_m.group(1).strip()

    # Social links: only real social platforms, NOT kickstarter.com links
    SOCIAL_DOMAINS = ("twitter.com", "x.com", "facebook.com", "instagram.com",
                      "youtube.com", "reddit.com", "linkedin.com", "tiktok.com")
    KS_SOCIAL = ("facebook.com/kickstarter", "instagram.com/kickstarter",
                 "twitter.com/kickstarter", "x.com/kickstarter")
    links = []
    seen_hrefs = set()
    for a in soup.select("a[href]"):
        href = a.get("href", "")
        label = _clean(a.get_text())
        if (any(d in href for d in SOCIAL_DOMAINS)
                and not any(ks in href for ks in KS_SOCIAL)
                and href not in seen_hrefs):
            seen_hrefs.add(href)
            links.append({"label": label or href, "href": href})

    # Creator website (non-social external links in the creator section)
    for a in soup.select("a[href]"):
        href = a.get("href", "")
        label = _clean(a.get_text())
        if (href.startswith("http") and "kickstarter.com" not in href
                and not any(d in href for d in SOCIAL_DOMAINS)
                and href not in seen_hrefs
                and label and len(label) < 80):
            seen_hrefs.add(href)
            links.append({"label": label, "href": href})
            break   # just one website

    return {
        "bio_text": bio_text,
        "created_projects": stats.get("created_projects", ""),
        "backed_projects":  stats.get("backed_projects", ""),
        "last_login":       stats.get("last_login", ""),
        "account_created":  stats.get("account_created", ""),
        "founder":          stats.get("founder", ""),
        "collaborators":    " | ".join(collaborators),
        "social_links":     " | ".join(f"{l['label']}: {l['href']}" for l in links),
    }

### `scrape_faqs`

In [25]:
async def scrape_faqs(page: Page, base: str) -> list:
    log.info("  → faqs /faqs")
    html = await _goto(page, base + "/faqs")
    soup = BeautifulSoup(html, "lxml")
    faqs = []

    full_text = soup.get_text("\n")
    faq_match = re.search(
        r"Frequently Asked Questions\n+(.*?)(?:Don.t see the answer|Ask a question|Similar projects)",
        full_text, re.DOTALL | re.IGNORECASE
    )
    if faq_match:
        faq_section = faq_match.group(1)
        entries = re.split(r"\nLast updated:\s*[^\n]+\n?", faq_section)
        for entry in entries:
            entry = entry.strip()
            if not entry or len(entry) < 10:
                continue
            lines = [l.strip() for l in entry.split("\n") if l.strip()]
            if not lines:
                continue
            q = lines[0]
            a = " ".join(lines[1:]) if len(lines) > 1 else ""
            if len(q) < 300 and ("?" in q or len(a) > 5):
                faqs.append({"question": q, "answer": a})

    # dt/dd fallback
    if not faqs:
        for dt in soup.find_all("dt"):
            dd = dt.find_next_sibling("dd")
            q = _clean(dt.get_text())
            if q and len(q) < 300:
                faqs.append({"question": q, "answer": _clean(dd.get_text()) if dd else ""})

    log.info("    → %d FAQs", len(faqs))
    return faqs


# FIX: click "Load more" until all updates visible; extract images/video per update

### `scrape_updates`
**Fix:** `_click_load_more()` until all updates loaded · `video_url` + `image_urls` per update

In [26]:
# FIX: click "Load more" until all updates visible; extract images/video per update
async def scrape_updates(page: Page, base: str) -> list:
    log.info("  → updates /posts (Load more until all visible)")
    updates = []

    # Try JSON API first (works for some/live projects)
    for _ in range(MAX_UPDATE_PAGES):
        url = base + "/posts.json"
        data = await _fetch_json_via_browser(page, url)
        posts = (data or {}).get("posts", (data or {}).get("updates", []))
        if posts:
            cursor = None
            for p in posts:
                body_html = p.get("body", "")
                imgs, vid = _extract_body_media(body_html)
                updates.append({
                    "id": p.get("id"), "sequence": p.get("sequence"),
                    "title": p.get("title"),
                    "body": _clean(body_html),
                    "published_at": _ts(p.get("published_at")),
                    "likes_count": p.get("likes_count"),
                    "comments_count": p.get("comments_count"),
                    "url": p.get("url", ""),
                    "video_url": vid,
                    "image_urls": " | ".join(imgs),
                })
            cursor = (data or {}).get("next_cursor")
            if not cursor:
                break
        else:
            break

    # HTML fallback: navigate to /posts and click Load more
    if not updates:
        await _goto(page, base + "/posts")
        log.info("    clicking Load more for updates...")
        clicked = await _click_load_more(page, MAX_UPDATE_PAGES, delay_ms=2500)
        log.info("    clicked Load more %d times", clicked)
        html = await page.content()
        soup = BeautifulSoup(html, "lxml")
        full_text = soup.get_text("\n")

        # Extract update blocks with regex
        update_blocks = re.findall(
            r"Update\s+#(\d+)\s+([^\n]+)\n(.*?)(?=Update\s+#\d+|Showing\s+\d+|$)",
            full_text, re.DOTALL
        )
        for seq, title, rest in update_blocks:
            rest = rest.strip()
            date_m = re.search(
                r"(January|February|March|April|May|June|July|August|September|"
                r"October|November|December)\s+\d{1,2},?\s+\d{4}", rest)
            published = date_m.group(0) if date_m else ""
            body = rest
            if date_m:
                body = rest[rest.find(published) + len(published):].strip()
            body = re.sub(r"\n?\d+\s+\d+\s*\n?Read more.*$", "", body, flags=re.DOTALL).strip()

            # Images from the update HTML snippet (within the page)
            # Find the matching update section in the soup
            seq_int = int(seq)
            update_sec = soup.find(string=re.compile(rf"Update\s*#\s*{seq_int}\b"))
            imgs, vid = [], ""
            if update_sec:
                parent = update_sec.find_parent()
                if parent:
                    imgs, vid = _extract_body_media(str(parent))

            if title.strip():
                updates.append({
                    "id": None, "sequence": seq_int,
                    "title": title.strip(),
                    "body": _clean(body),
                    "published_at": published,
                    "likes_count": None, "comments_count": None, "url": "",
                    "video_url": vid,
                    "image_urls": " | ".join(imgs),
                })

    log.info("    → %d updates", len(updates))
    return updates


# FIX: DOM-based comment extraction via page.evaluate + Load more clicking

### `scrape_comments`
**v5.2 card-first fix:** Starts from every `a[href*='/profile/']` link → climbs to smallest card ancestor → groups by DOM containment (top-level vs reply) → extracts own text only.

In [27]:
async def scrape_comments(page: Page, base: str) -> list:
    """
    Scrape comments from the /comments tab.

    Root cause of previous failure:
      The old walker selected the 'most link-dense' container, then walked only
      its DIRECT children.  But Kickstarter wraps each comment card in 1-2
      intermediate divs, so only 1 card (the wrapper) was returned.

    New approach — CARD-FIRST:
      1. Find every a[href*='/profile/'] link on the page.
      2. For each link, climb the DOM to find its 'card' — the smallest
         ancestor that both contains a <p> AND looks visually like a card
         (border / li / article / comment class, OR fixed height).
      3. Collect all unique cards into a flat set.
      4. Split into top-level cards vs reply cards using containment:
         a card is a reply if another card contains it.
      5. For each top-level card, extract own body text (exclude text inside
         nested reply cards), author, and timestamp.
      6. For each reply card, do the same and attach the parent card id.
    """
    log.info("  → comments")
    comments = []

    # ── 1. JSON API ──────────────────────────────────────────────────────────
    cursor = None
    for _ in range(MAX_COMMENT_PAGES):
        url = base + "/comments.json" + (f"?cursor={cursor}" if cursor else "")
        data = await _fetch_json_via_browser(page, url)
        raw = (data or {}).get("comments", [])
        if not raw:
            break
        for c in raw:
            author = c.get("author") or {}
            comments.append({
                "id":           c.get("id"),
                "body":         _clean(c.get("body", "")),
                "created_at":   _ts(c.get("created_at")),
                "deleted":      c.get("deleted", False),
                "author_name":  author.get("name", ""),
                "author_url":   (author.get("urls") or {}).get("web", {}).get("user", ""),
                "replies": [
                    {"id": r.get("id"), "body": _clean(r.get("body", "")),
                     "created_at": _ts(r.get("created_at")),
                     "author_name": (r.get("author") or {}).get("name", "")}
                    for r in c.get("replies", [])
                ],
            })
        cursor = (data or {}).get("next_cursor")
        if not cursor:
            break

    if comments:
        log.info("    → %d comments (JSON API)", len(comments))
        return comments

    # ── 2. Navigate to /comments and wait for React render ───────────────────
    log.info("    JSON API blocked — card-first DOM extraction")
    await page.goto(base + "/comments", wait_until="domcontentloaded", timeout=PAGE_TIMEOUT)
    try:
        await page.wait_for_selector('a[href*="/profile/"]', timeout=15_000)
    except Exception:
        log.warning("    timeout waiting for profile links")
    await page.wait_for_timeout(2_500)  # let React finish paint

    clicked = await _click_load_more(page, MAX_COMMENT_PAGES, delay_ms=2_000)
    log.info("    clicked Load more %d times", clicked)

    # ── 3. Card-first DOM extraction ─────────────────────────────────────────
    # The JS runs inside the browser so it has direct DOM access.
    CARD_FIRST_SCRIPT = """
    () => {
        // ── helpers ──────────────────────────────────────────────────────
        function isCardLike(el) {
            if (!el || el === document.body) return false;
            const tag = el.tagName;
            if (tag === 'LI' || tag === 'ARTICLE') return true;
            const cls = (el.className || '').toLowerCase();
            if (cls.includes('border') || cls.includes('card') ||
                cls.includes('comment') || cls.includes('thread')) return true;
            // Fallback: element has a visible border
            const style = window.getComputedStyle(el);
            const bw = parseInt(style.borderTopWidth) || 0;
            if (bw >= 1) return true;
            return false;
        }

        function findCardAncestor(link) {
            // Walk up from the profile link until we hit a card-like element
            // that also has at least one <p> descendant.
            let el = link.parentElement;
            let steps = 0;
            while (el && el !== document.body && steps < 12) {
                if (isCardLike(el) && el.querySelector('p')) return el;
                el = el.parentElement;
                steps++;
            }
            // Fallback: just find the closest ancestor with a <p>
            el = link.parentElement;
            while (el && el !== document.body) {
                if (el.querySelector('p')) return el;
                el = el.parentElement;
            }
            return null;
        }

        function getTimestamp(cardEl, excludeEls) {
            // Look for relative-time text outside of any excluded sub-elements
            const TIME_RE = /\\d+\\s+(second|minute|hour|day|month|year)/i;
            const nodes = cardEl.querySelectorAll('time, span');
            for (const n of nodes) {
                if (excludeEls.some(ex => ex.contains(n))) continue;
                const dt = n.getAttribute && n.getAttribute('datetime');
                if (dt) return dt;
                if (TIME_RE.test(n.innerText)) return n.innerText.trim();
            }
            return '';
        }

        function getBody(cardEl, excludeEls) {
            // Collect <p> text that does NOT live inside any excluded sub-element
            const ps = cardEl.querySelectorAll('p');
            const own = [];
            for (const p of ps) {
                if (!excludeEls.some(ex => ex.contains(p))) {
                    const t = p.innerText.trim();
                    if (t) own.push(t);
                }
            }
            return own.join('\\n');
        }

        // ── Step 1: collect every profile link on the page ────────────────
        const profileLinks = Array.from(
            document.querySelectorAll('a[href*="/profile/"]')
        );

        // ── Step 2: find card ancestor for each link; deduplicate ─────────
        const cardMap = new Map();   // card element → profile link
        profileLinks.forEach(link => {
            const card = findCardAncestor(link);
            if (card && !cardMap.has(card)) cardMap.set(card, link);
        });

        const allCards = Array.from(cardMap.keys());
        if (allCards.length === 0) return [];

        // ── Step 3: split top-level vs reply cards by containment ─────────
        // A card is a REPLY if any OTHER card strictly contains it.
        function isContained(cardA, allCards) {
            return allCards.some(other => other !== cardA && other.contains(cardA));
        }

        const topCards   = allCards.filter(c => !isContained(c, allCards));
        const replyCards = allCards.filter(c =>  isContained(c, allCards));

        // ── Step 4: build results ─────────────────────────────────────────
        const results = [];
        const seen = new Set();

        topCards.forEach((card, idx) => {
            const link = cardMap.get(card);
            // Replies nested inside THIS top-level card
            const ownReplies = replyCards.filter(rc => card.contains(rc));

            const body = getBody(card, ownReplies);
            if (!body) return;

            const key = (link.href || '') + body.slice(0, 30);
            if (seen.has(key)) return;
            seen.add(key);

            const cardId = card.id || card.dataset?.id || ('top_' + idx);
            results.push({
                id:               cardId,
                author_name:      link.innerText.trim(),
                author_url:       link.href,
                created_at:       getTimestamp(card, ownReplies),
                body:             body,
                parent_comment_id: ''
            });

            // Emit each reply with parent id
            ownReplies.forEach((rc, ri) => {
                const rlink = cardMap.get(rc);
                if (!rlink) return;
                // Sub-replies inside this reply (unlikely but safe)
                const subReplies = replyCards.filter(sr => rc.contains(sr) && sr !== rc);
                const rbody = getBody(rc, subReplies);
                if (!rbody) return;
                const rkey = (rlink.href || '') + rbody.slice(0, 30);
                if (seen.has(rkey)) return;
                seen.add(rkey);
                results.push({
                    id:               rc.id || rc.dataset?.id || (cardId + '_r' + ri),
                    author_name:      rlink.innerText.trim(),
                    author_url:       rlink.href,
                    created_at:       getTimestamp(rc, subReplies),
                    body:             rbody,
                    parent_comment_id: cardId
                });
            });
        });

        return results;
    }
    """

    try:
        raw = await page.evaluate(CARD_FIRST_SCRIPT)
        for c in (raw or []):
            if c.get("body") and c.get("author_name"):
                comments.append({
                    "id":               c.get("id", ""),
                    "body":             c["body"],
                    "created_at":       c.get("created_at", ""),
                    "deleted":          False,
                    "author_name":      c["author_name"],
                    "author_url":       c.get("author_url", ""),
                    "parent_comment_id": c.get("parent_comment_id", ""),
                    "replies":          [],
                })
        log.info("    card-first extractor found %d comments/replies", len(comments))
    except Exception as exc:
        log.warning("    DOM extraction failed: %s", exc)

    # ── 4. Plain-text last-resort fallback ────────────────────────────────────
    if not comments:
        log.info("    falling back to text extraction")
        html = await page.content()
        soup = BeautifulSoup(html, "lxml")
        full_text = soup.get_text("\n")
        TIME_PAT = (
            r"(?:\d+\s+(?:days?|hours?|minutes?|seconds?)\s+ago"
            r"|about\s+\d+\s+months?\s+ago|\d+\s+months?\s+ago)"
        )
        sec_m = re.search(
            r"Only backers can post comments.*?\n(.*?)(?:Showing \d+|Similar projects|$)",
            full_text, re.DOTALL | re.IGNORECASE
        )
        if sec_m:
            parts = re.findall(
                rf"(.{{1,100}})\n({TIME_PAT})\n(.*?)(?=\n.{{1,100}}\n{TIME_PAT}\n|$)",
                sec_m.group(1), re.DOTALL
            )
            for author, ts, body in parts:
                author = author.strip()
                body = re.sub(r"\n?\d+\s*\n?Read more.*$", "", body.strip(), flags=re.DOTALL).strip()
                if author and body:
                    comments.append({
                        "id": None, "body": _clean(body),
                        "created_at": ts.strip(), "deleted": False,
                        "author_name": author, "author_url": "",
                        "parent_comment_id": "", "replies": [],
                    })

    log.info("    → %d comments total", len(comments))
    return comments

### `scrape_community`

In [28]:
async def scrape_community(page: Page, base: str) -> dict:
    log.info("  → community /community")
    html = await _goto(page, base + "/community")
    soup = BeautifulSoup(html, "lxml")
    full_text = soup.get_text("\n")

    total_backers = ""
    m = re.search(r"([\d,]+)\s+people\s+are\s+supporting", full_text)
    if m:
        total_backers = m.group(1).replace(",", "")

    new_backers = ""
    m_new = re.search(r"([\d,]+)\s*\n?\s*backers\s+had\s+never\s+backed", full_text)
    if m_new:
        new_backers = m_new.group(1).replace(",", "")

    returning_backers = ""
    m_ret = re.search(r"([\d,]+)\s*\n?\s*backers\s+had\s+backed\s+a\s+project\s+on\s+Kickstarter\s+before", full_text)
    if m_ret:
        returning_backers = m_ret.group(1).replace(",", "")

    top_cities = []
    city_m = re.search(r"Top Cities(.*?)(?:Top Countries|Where Backers Come From)", full_text, re.DOTALL)
    if city_m:
        city_entries = re.findall(
            r"([A-Z][a-zA-Z\s\-]+?)\s+(?:United States|United Kingdom|Australia|Canada|Germany|"
            r"Netherlands|France|Sweden|Denmark|Norway|[A-Z][a-z][\w\s]*?)\s+([\d,]+)\s+backers?",
            city_m.group(1)
        )
        top_cities = [{"city": c[0].strip(), "backers": c[1].replace(",", "")} for c in city_entries[:10]]

    top_countries = []
    country_m = re.search(r"Top Countries(.*?)(?:New Backers|Returning Backers|$)", full_text, re.DOTALL)
    if country_m:
        country_entries = re.findall(
            r"(United States|United Kingdom|Canada|Australia|Germany|Netherlands|France|"
            r"Sweden|Denmark|Norway|[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\s+([\d,]+)\s+backers?",
            country_m.group(1)
        )
        top_countries = [{"country": c[0].strip(), "backers": c[1].replace(",", "")} for c in country_entries[:10]]

    collaborators = re.findall(r"Collaborator\s*\n?\s*([^\n]+)", full_text, re.IGNORECASE)
    collaborators = [c.strip() for c in collaborators if c.strip() and len(c.strip()) < 80]

    return {
        "total_backers": total_backers,
        "new_backers": new_backers,
        "returning_backers": returning_backers,
        "top_cities": top_cities,
        "top_countries": top_countries,
        "collaborators": collaborators,
    }

## 6 · Main orchestrator

In [29]:
# ── Main orchestrator ─────────────────────────────────────────────────────────

async def _scrape_all(url: str) -> dict:
    creator_slug, project_slug = slug_from_url(url)
    base = f"https://www.kickstarter.com/projects/{creator_slug}/{project_slug}"
    log.info("Scraping: %s", base)
    async with async_playwright() as pw:
        browser, page = await _make_browser_context(pw)
        try:
            return {
                "meta":      await scrape_meta(page, base),
                "campaign":  await scrape_campaign(page, base),
                "rewards":   await scrape_rewards(page, base),
                "creator":   await scrape_creator(page, base),
                "faqs":      await scrape_faqs(page, base),
                "updates":   await scrape_updates(page, base),
                "comments":  await scrape_comments(page, base),
                "community": await scrape_community(page, base),
            }
        finally:
            await browser.close()


def scrape_project(url: str) -> dict:
    return run_pw(_scrape_all(url))

## 7 · CSV export

All 8 CSVs with flat, clean columns — no Python dict strings.

In [30]:
# ── CSV export ────────────────────────────────────────────────────────────────

import csv

def export_to_csv(all_data: dict, csv_dir: Path):
    csv_dir.mkdir(parents=True, exist_ok=True)

    def write_csv(path, rows, fieldnames=None):
        if not rows:
            print(f"  (empty) {path.name}"); return
        if fieldnames is None:
            seen, fieldnames = set(), []
            for row in rows:
                for k in row:
                    if k not in seen:
                        fieldnames.append(k); seen.add(k)
        with open(path, "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
            w.writeheader(); w.writerows(rows)
        print(f"  ✓ {path.name:<42} {len(rows):>5} rows")

    # 1. META — clean numbers, no dicts
    META_FIELDS = [
        "project_key","title","blurb","project_url","creator_slug","project_slug",
        "creator_name","pledged","usd_pledged","goal","percent_funded","backers_count",
        "currency","currency_symbol","state","location","category",
        "funding_period","last_updated",
    ]
    write_csv(csv_dir / "meta.csv",
              [{"project_key": k, **v.get("meta", {})} for k, v in all_data.items()],
              META_FIELDS)

    # 2. CAMPAIGN — story + risks + video + images (no stats dict, no raw HTML)
    CAMPAIGN_FIELDS = ["project_key","story_text","risks_text","video_url","image_urls","image_count"]
    write_csv(csv_dir / "campaign.csv",
              [{"project_key": k, **{f: v.get("campaign", {}).get(f, "")
                for f in CAMPAIGN_FIELDS[1:]}} for k, v in all_data.items()],
              CAMPAIGN_FIELDS)

    # 3. REWARDS — title and description now separate
    REWARD_FIELDS = ["project_key","id","amount","title","description",
                     "backers_count","estimated_delivery","ships_to","limit"]
    write_csv(csv_dir / "rewards.csv",
              [{"project_key": k, **r} for k, v in all_data.items()
               for r in v.get("rewards", [])],
              REWARD_FIELDS)

    # 4. CREATOR — flat fields
    CREATOR_FIELDS = ["project_key","bio_text","created_projects","backed_projects",
                      "last_login","account_created","founder","collaborators","social_links"]
    write_csv(csv_dir / "creator.csv",
              [{"project_key": k, **{f: v.get("creator", {}).get(f, "")
                for f in CREATOR_FIELDS[1:]}} for k, v in all_data.items()],
              CREATOR_FIELDS)

    # 5. FAQs
    write_csv(csv_dir / "faqs.csv", [
        {"project_key": k, "faq_number": i,
         "question": f.get("question",""), "answer": f.get("answer","")}
        for k, v in all_data.items()
        for i, f in enumerate(v.get("faqs", []), 1)
    ])

    # 6. UPDATES — with video_url and image_urls columns
    UPDATE_FIELDS = ["project_key","id","sequence","title","published_at",
                     "likes_count","comments_count","body","url","video_url","image_urls"]
    write_csv(csv_dir / "updates.csv",
              [{"project_key": k, **u} for k, v in all_data.items()
               for u in v.get("updates", [])],
              UPDATE_FIELDS)

    # 7. COMMENTS — flatten replies as child rows
    COMMENT_FIELDS = ["project_key","id","parent_comment_id","author_name",
                      "author_url","created_at","deleted","body"]
    comment_rows = []
    for k, v in all_data.items():
        for c in v.get("comments", []):
            pid = c.get("parent_comment_id", "")
            comment_rows.append({
                "project_key": k, "id": c.get("id",""),
                "parent_comment_id": pid,
                "author_name": c.get("author_name",""), "author_url": c.get("author_url",""),
                "created_at": c.get("created_at",""), "deleted": c.get("deleted", False),
                "body": c.get("body",""),
            })
            for reply in c.get("replies", []):
                comment_rows.append({
                    "project_key": k, "id": reply.get("id",""),
                    "parent_comment_id": c.get("id",""),
                    "author_name": reply.get("author_name",""), "author_url": "",
                    "created_at": reply.get("created_at",""), "deleted": False,
                    "body": reply.get("body",""),
                })
    write_csv(csv_dir / "comments.csv", comment_rows, COMMENT_FIELDS)

    # 8. COMMUNITY — top_cities / top_countries as JSON strings for readability
    COMMUNITY_FIELDS = ["project_key","total_backers","new_backers","returning_backers",
                        "top_cities","top_countries","collaborators"]
    community_rows = []
    for k, v in all_data.items():
        comm = v.get("community", {})
        community_rows.append({
            "project_key": k,
            "total_backers": comm.get("total_backers",""),
            "new_backers": comm.get("new_backers",""),
            "returning_backers": comm.get("returning_backers",""),
            "top_cities": json.dumps(comm.get("top_cities",[])),
            "top_countries": json.dumps(comm.get("top_countries",[])),
            "collaborators": " | ".join(comm.get("collaborators",[])),
        })
    write_csv(csv_dir / "community.csv", community_rows, COMMUNITY_FIELDS)

    print(f"\n✓ All CSVs → {csv_dir.resolve()}")

## 8 · Scrape a project

In [31]:
CUSTOM_URL = "https://www.kickstarter.com/projects/elanlee/exploding-kittens"

data = scrape_project(CUSTOM_URL)

m = data["meta"]
print(f"Title       : {m['title']}")
print(f"Pledged     : ${int(m['pledged']):,}" if m.get("pledged") else "Pledged: ?")
print(f"Goal        : ${int(m['goal']):,}" if m.get("goal") else "Goal: ?")
print(f"% Funded    : {m['percent_funded']}")
print(f"Backers     : {int(m['backers_count']):,}" if m.get("backers_count") else "")
print(f"Category    : {m['category']}")
print(f"Location    : {m['location']}")
print(f"Period      : {m['funding_period']}")
print()
print(f"Video URL   : {data['campaign'].get('video_url','')[:80] or '(none)'}")
print(f"Story imgs  : {data['campaign'].get('image_count', 0)}")
print(f"Rewards     : {len(data['rewards'])}")
print(f"FAQs        : {len(data['faqs'])}")
print(f"Updates     : {len(data['updates'])}")
print(f"Comments    : {len(data['comments'])}")
print(f"Community   : new={data['community'].get('new_backers',0)} returning={data['community'].get('returning_backers',0)}")

13:26:43  INFO      Scraping: https://www.kickstarter.com/projects/elanlee/exploding-kittens
13:26:52  INFO        → meta (HTML)
13:27:38  INFO          title=Exploding Kittens pledged=8782571 goal=10000 backers=219382
13:27:38  INFO        → campaign /description
13:28:12  INFO        → rewards /rewards
13:28:40  INFO          → 4 reward tiers
13:28:40  INFO        → creator /creator
13:29:46  INFO        → faqs /faqs
13:31:04  INFO          → 19 FAQs
13:31:04  INFO        → updates /posts (Load more until all visible)
13:33:45  INFO          clicking Load more for updates...
13:35:34  INFO          clicked Load more 1 times
13:36:07  INFO          → 20 updates
13:36:07  INFO        → comments
13:36:11  INFO          JSON API blocked — card-first DOM extraction
13:36:35  WARNING       timeout waiting for profile links
13:36:44  INFO          clicked Load more 0 times
13:36:52  INFO          card-first extractor found 0 comments/replies
13:36:52  INFO          falling back to text extr

Title       : Exploding Kittens
Pledged     : $8,782,571
Goal        : $10,000
% Funded    : 87,826%
Backers     : 219,382
Category    : Tabletop Games
Location    : Los Angeles, CA
Period      : Jan 20 2015 - Feb 19 2015

Video URL   : https://v2.kickstarter.com/1779942666-7o%2BUW5tTuivysfckr1bRrSrIrNb%2BK1%2BswsSn
Story imgs  : 13
Rewards     : 4
FAQs        : 19
Updates     : 20
Comments    : 27
Community   : new=95583 returning=123799


## 9 · Save JSON & export 8 CSVs

In [34]:
import json
from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

key = data["meta"].get("project_slug", "project")
jp = OUTPUT_DIR / f"{key}.json"
with open(jp, "w", encoding="utf-8") as fh:
    json.dump(data, fh, indent=2, default=str)
print(f"JSON saved: {jp}")

export_to_csv({key: data}, OUTPUT_DIR / "csv")

JSON saved: output\exploding-kittens.json
  ✓ meta.csv                                       1 rows
  ✓ campaign.csv                                   1 rows
  ✓ rewards.csv                                    4 rows
  ✓ creator.csv                                    1 rows
  ✓ faqs.csv                                      19 rows
  ✓ updates.csv                                   20 rows
  ✓ comments.csv                                  27 rows
  ✓ community.csv                                  1 rows

✓ All CSVs → C:\Users\Chan1919\Downloads\Research Project\output\csv
